# Clasificación de irradiancia con funciones Kernel

**Objetivo.** Comparar un conjunto de pipelines para clasificar irradiancia a partir de los datasets Landsat y MODIS, evaluar cada configuración con métricas comunes y exportar los modelos seleccionados para una aplicación web.

> Esta versión tiene una organización propia y está preparada para ejecutarse en Google Colab. Primero se cargan y revisan los datos, luego se implementan los clasificadores Kernel, se construye el campo experimental, se validan las configuraciones y finalmente se evalúan y exportan los modelos.

**Importante:** la primera ejecución usa un modo de prueba para verificar que todas las piezas funcionan. Después de comprobarlo, se cambia una sola variable a `RUN_FULL_GRID = True` para ejecutar las 648 configuraciones por dataset.


## Punto 1. Datos Landsat y MODIS

Los archivos se obtienen de las rutas indicadas en la actividad. Si Google Colab no permite acceder a GitHub, también se pueden subir manualmente los dos CSV a la carpeta `data/`.

Las variables predictoras son `latitude`, `longitude` y `band1` a `band7`; la variable objetivo continua es `value` (irradiancia).

In [ ]:
# Instalación mínima para Colab
!pip -q install numpy pandas scikit-learn matplotlib seaborn joblib folium streamlit streamlit-folium

In [ ]:
from pathlib import Path
import itertools, time, warnings, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

URLS = {
    'landsat': 'https://raw.githubusercontent.com/magohector/fkernel/master/Experimentos/landsat_model.csv',
    'modis': 'https://raw.githubusercontent.com/magohector/fkernel/master/Experimentos/modis_model.csv',
}

for name, url in URLS.items():
    target = DATA_DIR / f'{name}_model.csv'
    if not target.exists():
        try:
            pd.read_csv(url).to_csv(target, index=False)
            print('Descargado:', target)
        except Exception as e:
            print(f'No se pudo descargar {name}. Sube {target.name} manualmente a /content/data. Error: {e}')

landsat = pd.read_csv(DATA_DIR/'landsat_model.csv')
modis = pd.read_csv(DATA_DIR/'modis_model.csv')
DATASETS = {'landsat': landsat, 'modis': modis}

FEATURES = ['latitude','longitude','band1','band2','band3','band4','band5','band6','band7']
TARGET = 'value'
for name, df in DATASETS.items():
    print(name.upper(), df.shape)
    display(df.head())
    print('Nulos:', int(df[FEATURES+[TARGET]].isna().sum().sum()))

### Exploración inicial

Se revisa la distribución de la irradiancia y la ubicación geográfica de las observaciones. Esto permite comprobar que los dos datasets tienen la estructura necesaria antes de entrenar.

In [ ]:
for name, df in DATASETS.items():
    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(df[TARGET].dropna(), bins=30)
    ax.set_title(f'Distribución de irradiancia - {name.upper()}')
    ax.set_xlabel('Irradiancia')
    ax.set_ylabel('Frecuencia')
    plt.show()

for name, df in DATASETS.items():
    plt.figure(figsize=(6,5))
    plt.scatter(df['longitude'], df['latitude'], c=df[TARGET], s=8)
    plt.title(f'Ubicación de observaciones - {name.upper()}')
    plt.xlabel('Longitud'); plt.ylabel('Latitud'); plt.colorbar(label='Irradiancia')
    plt.show()

## Punto 2. KSVC y KANNC

Se incorporan las clases solicitadas y las funciones Kernel que necesitan. Se usan nueve kernels para construir el campo experimental: lineal, polinomial, RBF, hiperbólico, triangular, radial básico, racional cuadrático, Canberra y truncado.

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import RidgeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_array, check_X_y, check_is_fitted

# ------------------------------------------------------------------
# Funciones Kernel del experimento
# ------------------------------------------------------------------
def linear(x, y, degree=2, gamma=0.1, coef0=0.1):
    return float(np.dot(x, y))


def polynomial(x, y, degree=2, gamma=0.1, coef0=0.1):
    return float((gamma * np.dot(x, y) + coef0) ** degree)


def rbf(x, y, degree=2, gamma=0.1, coef0=0.1):
    return float(np.exp(-gamma * np.sum((x - y) ** 2)))


def hyperbolic(x, y, degree=2, gamma=0.1, coef0=0.1):
    return float(np.tanh(gamma * np.dot(x, y) + coef0))


def triangle(x, y, degree=2, gamma=0.1, coef0=0.1):
    d = np.linalg.norm(x - y)
    return float(max(0.0, 1.0 - d / max(gamma, 1e-12)))


def radial_basic(x, y, degree=2, gamma=0.1, coef0=0.1):
    return float(np.sum(np.exp(-gamma * (x - y) ** 2)) ** degree)


def rquadratic(x, y, degree=2, gamma=0.1, coef0=0.1):
    d2 = np.sum((x - y) ** 2)
    a = max(abs(coef0), 1e-8)
    return float(1.0 - d2 / (d2 + a))


def canberra(x, y, degree=2, gamma=0.1, coef0=0.1):
    den = np.abs(x) + np.abs(y)
    term = np.divide(np.abs(x - y), den, out=np.zeros_like(x, dtype=float), where=den != 0)
    return float(1.0 - gamma * np.mean(term))


def truncated(x, y, degree=2, gamma=0.1, coef0=0.1):
    d = np.abs(x - y)
    return float(np.mean(np.maximum(1.0 - d / max(gamma, 1e-12), 0.0)))


KERNEL_FUNCTIONS = {
    'linear': linear,
    'poly': polynomial,
    'rbf': rbf,
    'hyperbolic': hyperbolic,
    'triangle': triangle,
    'radial_basic': radial_basic,
    'rquadratic': rquadratic,
    'canberra': canberra,
    'truncated': truncated,
}


def gram_matrix(X, Y, kernel_name, degree=2, gamma=0.1, coef0=0.1):
    """Construye la matriz Gram K(X,Y) de forma vectorizada."""
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    if kernel_name == 'linear':
        K = X @ Y.T
    elif kernel_name == 'poly':
        K = (gamma * (X @ Y.T) + coef0) ** degree
    elif kernel_name == 'rbf':
        x2 = np.sum(X * X, axis=1)[:, None]
        y2 = np.sum(Y * Y, axis=1)[None, :]
        d2 = np.maximum(x2 + y2 - 2.0 * (X @ Y.T), 0.0)
        K = np.exp(-gamma * d2)
    elif kernel_name == 'hyperbolic':
        K = np.tanh(gamma * (X @ Y.T) + coef0)
    elif kernel_name == 'triangle':
        d = np.sqrt(np.maximum(
            np.sum(X * X, axis=1)[:, None]
            + np.sum(Y * Y, axis=1)[None, :]
            - 2.0 * (X @ Y.T), 0.0
        ))
        K = np.maximum(1.0 - d / max(gamma, 1e-12), 0.0)
    elif kernel_name == 'radial_basic':
        # Suma de exponenciales por dimensión, elevada al grado.
        K = np.sum(np.exp(-gamma * (X[:, None, :] - Y[None, :, :]) ** 2), axis=2) ** degree
    elif kernel_name == 'rquadratic':
        d2 = np.maximum(
            np.sum(X * X, axis=1)[:, None]
            + np.sum(Y * Y, axis=1)[None, :]
            - 2.0 * (X @ Y.T), 0.0
        )
        a = max(abs(coef0), 1e-8)
        K = 1.0 - d2 / (d2 + a)
    elif kernel_name == 'canberra':
        num = np.abs(X[:, None, :] - Y[None, :, :])
        den = np.abs(X[:, None, :]) + np.abs(Y[None, :, :])
        term = np.divide(num, den, out=np.zeros_like(num), where=den != 0)
        K = 1.0 - gamma * np.mean(term, axis=2)
    elif kernel_name == 'truncated':
        d = np.abs(X[:, None, :] - Y[None, :, :])
        K = np.mean(np.maximum(1.0 - d / max(gamma, 1e-12), 0.0), axis=2)
    else:
        raise ValueError(f'Kernel no reconocido: {kernel_name}')

    return np.nan_to_num(K, nan=0.0, posinf=1e12, neginf=-1e12)


def kernel_callable(kernel_name, degree=2, gamma=0.1, coef0=0.1):
    def call(X, Y):
        return gram_matrix(X, Y, kernel_name, degree, gamma, coef0)
    return call


def scalar_kernel_callable(kernel_name, degree=2, gamma=0.1, coef0=0.1):
    fn = KERNEL_FUNCTIONS[kernel_name]
    def call(x, y):
        return fn(np.asarray(x, dtype=float), np.asarray(y, dtype=float), degree, gamma, coef0)
    return call


class KSVC(SVC):
    """Adaptación de SVC para aceptar los kernels alternativos del experimento."""
    def __init__(self, C=1.0, kernel='rbf', degree=2, gamma=0.1, coef0=0.1,
                 shrinking=True, probability=False, tol=1e-3, cache_size=200,
                 class_weight=None, verbose=False, max_iter=-1,
                 decision_function_shape='ovr', random_state=2021):
        super().__init__(C=C, kernel=kernel, degree=degree, gamma=gamma, coef0=coef0,
                         shrinking=shrinking, probability=probability, tol=tol,
                         cache_size=cache_size, class_weight=class_weight,
                         verbose=verbose, max_iter=max_iter,
                         decision_function_shape=decision_function_shape,
                         random_state=random_state)
        self._kernel_name = kernel

    def fit(self, X, y, sample_weight=None):
        requested = self.kernel
        if requested in KERNEL_FUNCTIONS and requested not in {'linear', 'poly', 'rbf'}:
            # SVC necesita conservar el callable también durante predict/decision_function.
            self.kernel = kernel_callable(requested, self.degree, self.gamma, self.coef0)
        return super().fit(X, y, sample_weight=sample_weight)

    def decision_function(self, X):
        score = super().decision_function(X)
        if np.ndim(score) == 1:
            return np.column_stack([-score, score])
        return score


class KANNC(MLPClassifier):
    """MLP con expansión de características mediante Nystroem para el kernel elegido."""
    def __init__(self, hidden_layer_sizes=(50,), activation='identity', solver='adam',
                 alpha=0.0001, batch_size='auto', learning_rate='constant',
                 learning_rate_init=0.001, max_iter=300, shuffle=True,
                 random_state=2021, tol=1e-4, early_stopping=False,
                 validation_fraction=0.1, n_iter_no_change=10, kernel='rbf',
                 degree=2, gamma=0.1, coef0=0.1, n_components=60):
        super().__init__(hidden_layer_sizes=hidden_layer_sizes, activation=activation,
                         solver=solver, alpha=alpha, batch_size=batch_size,
                         learning_rate=learning_rate, learning_rate_init=learning_rate_init,
                         max_iter=max_iter, shuffle=shuffle, random_state=random_state,
                         tol=tol, early_stopping=early_stopping,
                         validation_fraction=validation_fraction,
                         n_iter_no_change=n_iter_no_change)
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.n_components = n_components
        self.feature_map_ = None

    def fit(self, X, y):
        self.feature_map_ = None
        if self.kernel != 'linear':
            n_components = min(self.n_components, len(X))
            self.feature_map_ = Nystroem(
                kernel=scalar_kernel_callable(self.kernel, self.degree, self.gamma, self.coef0),
                n_components=max(2, n_components),
                random_state=self.random_state
            )
            X = self.feature_map_.fit_transform(X)
        return super().fit(X, y)

    def _map(self, X):
        return self.feature_map_.transform(X) if self.feature_map_ is not None else X

    def predict(self, X):
        return super().predict(self._map(X))

    def predict_proba(self, X):
        return super().predict_proba(self._map(X))

    def decision_function(self, X):
        # Para AUC se usan probabilidades de clase.
        return self.predict_proba(X)


class KRidgeClassifier(RidgeClassifier):
    """RidgeClassifier extendido con formulación dual mediante una matriz Kernel."""
    def __init__(self, alpha=1.0, kernel='rbf', degree=2, gamma=0.1,
                 coef0=0.1, random_state=2021):
        super().__init__(alpha=alpha)
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.random_state = random_state
        self._classes = None

    @property
    def classes_(self):
        # Evita el conflicto de versiones de scikit-learn con classes_.
        return self._classes

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self._classes, encoded = np.unique(y, return_inverse=True)
        if len(self._classes) < 2:
            raise ValueError('Se necesitan al menos dos clases.')

        self.X_fit_ = X
        K = gram_matrix(X, X, self.kernel, self.degree, self.gamma, self.coef0)

        target = -np.ones((len(y), len(self._classes)))
        target[np.arange(len(y)), encoded] = 1.0

        A = K + self.alpha * np.eye(len(K))
        try:
            self.dual_coef_ = np.linalg.solve(A, target)
        except np.linalg.LinAlgError:
            self.dual_coef_ = np.linalg.pinv(A) @ target

        self.n_features_in_ = X.shape[1]
        return self

    def decision_function(self, X):
        check_is_fitted(self, ['dual_coef_', 'X_fit_'])
        X = check_array(X)
        K = gram_matrix(X, self.X_fit_, self.kernel, self.degree, self.gamma, self.coef0)
        scores = K @ self.dual_coef_
        return scores

    def predict(self, X):
        scores = self.decision_function(X)
        if scores.ndim == 1:
            scores = scores[:, None]
        return self.classes_[np.argmax(scores, axis=1)]

    def predict_proba(self, X):
        scores = self.decision_function(X)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        scores = scores - scores.max(axis=1, keepdims=True)
        exp_scores = np.exp(np.clip(scores, -50, 50))
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)


### Comprobación de KSVC y KANNC

In [ ]:
X_demo=landsat[FEATURES].to_numpy(float)
y_demo=(landsat[TARGET] >= landsat[TARGET].median()).astype(int).to_numpy()
ksvc_demo=KSVC(kernel='rbf', C=1.0, gamma=0.1)
ksvc_demo.fit(X_demo, y_demo)
kannc_demo=KANNC(kernel='rbf', gamma=0.1, max_iter=100)
kannc_demo.fit(X_demo, y_demo)
print('Clase KSVC:', type(ksvc_demo).__name__)
print('Predicciones KSVC:', ksvc_demo.predict(X_demo[:5]))
print('Predicciones KANNC:', kannc_demo.predict(X_demo[:5]))

## Punto 3. KRidgeClassifier con truco Kernel

`KRidgeClassifier` extiende `RidgeClassifier`, pero en lugar de trabajar directamente con las variables originales construye la matriz Gram `K(X, X)` mediante el kernel seleccionado. Para varias clases se usa una codificación one-vs-rest y se resuelve la ecuación regularizada en el espacio dual.

In [ ]:
# La clase KRidgeClassifier ya fue definida en la celda anterior.
# Esta celda comprueba que la propiedad classes_ sea de solo lectura y funcione
# correctamente con la versión de scikit-learn utilizada por Google Colab.
print('KRidgeClassifier listo:', KRidgeClassifier.__name__)
print('Hereda de RidgeClassifier:', issubclass(KRidgeClassifier, RidgeClassifier))


In [ ]:
kr_demo=KRidgeClassifier(kernel='rbf', alpha=1.0, gamma=0.1)
kr_demo.fit(X_demo, y_demo)
print('Clases:', kr_demo.classes_)
print('Predicciones:', kr_demo.predict(X_demo[:5]))

## Punto 4. Campo completo de pipelines

Se genera el producto cartesiano solicitado:

- 3 escaladores: StandardScaler, MinMaxScaler, Normalizer.
- 4 discretizadores de `value`: Uniform, K-Means, DBSCAN y jerárquico.
- 2 reductores: PCA y LDA.
- 3 modelos: KSVC, KANNC y KRidgeClassifier.
- 9 kernels.

Por tanto: **3 × 4 × 2 × 3 × 9 = 648 configuraciones por dataset** y **1296 configuraciones para Landsat + MODIS**.

In [ ]:
from dataclasses import dataclass
from itertools import product
import numpy as np
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score

SCALERS = ('standard', 'minmax', 'normalizer')
DISCRETIZERS = ('uniform', 'kmeans', 'dbscan', 'agglomerative')
REDUCERS = ('pca', 'lda')
MODELS = ('ksvc', 'kannc', 'kridge')
KERNELS = ('linear', 'poly', 'rbf', 'hyperbolic', 'triangle', 'radial_basic', 'rquadratic', 'canberra', 'truncated')

@dataclass(frozen=True)
class Configuration:
    scaler: str
    discretizer: str
    reducer: str
    model: str
    kernel: str

    @property
    def name(self):
        return '-'.join([self.scaler, self.discretizer, self.reducer, self.model, self.kernel])


def all_configurations():
    return [Configuration(*x) for x in product(SCALERS, DISCRETIZERS, REDUCERS, MODELS, KERNELS)]


class TargetDiscretizer:
    """Convierte la irradiancia continua en clases, ajustándose solo con entrenamiento."""
    def __init__(self, method, n_classes=5, random_state=2021):
        self.method = method
        self.n_classes = n_classes
        self.random_state = random_state

    def fit(self, y):
        y = np.asarray(y, dtype=float).ravel()
        if not np.isfinite(y).all():
            raise ValueError('La variable objetivo contiene valores no finitos.')
        if np.ptp(y) == 0:
            raise ValueError('La irradiancia no tiene variación.')

        k = min(self.n_classes, len(np.unique(y)))
        if k < 2:
            raise ValueError('No existen suficientes valores distintos para formar clases.')

        if self.method == 'uniform':
            edges = np.linspace(y.min(), y.max(), k + 1)
            self.centers_ = (edges[:-1] + edges[1:]) / 2

        elif self.method == 'kmeans':
            km = KMeans(n_clusters=k, n_init=10, random_state=self.random_state)
            km.fit(y[:, None])
            self.centers_ = km.cluster_centers_.ravel()

        elif self.method == 'agglomerative':
            ag = AgglomerativeClustering(n_clusters=k)
            labels = ag.fit_predict(y[:, None])
            self.centers_ = np.array([y[labels == c].mean() for c in np.unique(labels)])

        elif self.method == 'dbscan':
            std = y.std()
            z = ((y - y.mean()) / (std if std > 0 else 1.0))[:, None]
            neighbors = min(5, len(y))
            distances = NearestNeighbors(n_neighbors=neighbors).fit(z).kneighbors(z)[0][:, -1]
            candidates = np.unique(np.quantile(distances, np.linspace(.50, .95, 12)))
            best = None
            for eps in candidates:
                labels = DBSCAN(eps=float(eps), min_samples=min(5, len(y))).fit_predict(z)
                valid = [c for c in np.unique(labels) if c != -1]
                if len(valid) >= 2:
                    score = (abs(len(valid) - k), int(np.sum(labels == -1)))
                    if best is None or score < best[0]:
                        best = (score, valid, labels)
            if best is None:
                # Respaldo determinista para no perder el experimento por una partición difícil.
                km = KMeans(n_clusters=k, n_init=10, random_state=self.random_state)
                km.fit(y[:, None])
                self.centers_ = km.cluster_centers_.ravel()
            else:
                self.centers_ = np.array([y[best[2] == c].mean() for c in best[1]])
        else:
            raise ValueError(f'Discretizador no reconocido: {self.method}')

        self.centers_ = np.sort(np.unique(self.centers_))
        if len(self.centers_) < 2:
            raise ValueError('El discretizador produjo menos de dos clases.')
        return self

    def transform(self, y):
        y = np.asarray(y, dtype=float).ravel()
        return np.argmin(np.abs(y[:, None] - self.centers_[None, :]), axis=1)

    def fit_transform(self, y):
        return self.fit(y).transform(y)


def make_discretizer(name):
    return TargetDiscretizer(name)


def make_model(config):
    common = dict(degree=2, gamma=0.1, coef0=0.1, random_state=2021)
    if config.model == 'ksvc':
        return KSVC(C=1.0, kernel=config.kernel, **common)
    if config.model == 'kannc':
        return KANNC(kernel=config.kernel, **common)
    if config.model == 'kridge':
        return KRidgeClassifier(alpha=1.0, kernel=config.kernel, **common)
    raise ValueError(config.model)


def make_feature_pipeline(config):
    scalers = {
        'standard': StandardScaler(),
        'minmax': MinMaxScaler(),
        'normalizer': Normalizer()
    }
    if config.reducer == 'pca':
        reducer = PCA(n_components=0.95, random_state=2021)
    else:
        reducer = LinearDiscriminantAnalysis()
    return Pipeline([
        ('scaler', scalers[config.scaler]),
        ('reducer', reducer),
        ('model', make_model(config))
    ])


class IrradiancePipeline:
    def __init__(self, config):
        self.config = config

    def fit(self, X, y):
        self.discretizer_ = make_discretizer(self.config.discretizer)
        labels = self.discretizer_.fit_transform(y)
        self.pipeline_ = make_feature_pipeline(self.config)
        self.pipeline_.fit(X, labels)
        self.classes_ = np.asarray(self.pipeline_.named_steps['model'].classes_)
        return self

    def predict(self, X):
        return self.pipeline_.predict(X)

    def decision_function(self, X):
        model = self.pipeline_.named_steps['model']
        if hasattr(model, 'decision_function'):
            scores = self.pipeline_.decision_function(X)
        else:
            scores = self.pipeline_.predict_proba(X)
        scores = np.asarray(scores)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        return scores

    def transform_target(self, y):
        return self.discretizer_.transform(y)


def safe_auc(y_true, scores, classes):
    values = []
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    for j, c in enumerate(classes):
        binary = (y_true == c).astype(int)
        if len(np.unique(binary)) == 2 and j < scores.shape[1]:
            values.append(roc_auc_score(binary, scores[:, j]))
    return float(np.mean(values)) if values else np.nan


def metrics(y_true, pred, scores, classes):
    return {
        'accuracy': accuracy_score(y_true, pred),
        'f1_macro': f1_score(y_true, pred, average='macro', zero_division=0),
        'auc_ovr': safe_auc(y_true, scores, classes),
        'mcc': matthews_corrcoef(y_true, pred)
    }


def evaluate(config, X, y, train_idx, test_idx, seed=2021):
    Xtr, ytr = X[train_idx], y[train_idx]
    kf = KFold(n_splits=3, shuffle=True, random_state=seed)
    rows = []

    for fit_idx, val_idx in kf.split(Xtr):
        estimator = IrradiancePipeline(config).fit(Xtr[fit_idx], ytr[fit_idx])
        y_val = estimator.transform_target(ytr[val_idx])
        pred = estimator.predict(Xtr[val_idx])
        scores = estimator.decision_function(Xtr[val_idx])
        rows.append(metrics(y_val, pred, scores, estimator.classes_))

    cv = {}
    for key in rows[0]:
        values = np.asarray([r[key] for r in rows], dtype=float)
        cv[f'cv_{key}_mean'] = float(np.nanmean(values))
        cv[f'cv_{key}_std'] = float(np.nanstd(values, ddof=1))

    estimator = IrradiancePipeline(config).fit(Xtr, ytr)
    y_test = estimator.transform_target(y[test_idx])
    pred_test = estimator.predict(X[test_idx])
    scores_test = estimator.decision_function(X[test_idx])
    hold = metrics(y_test, pred_test, scores_test, estimator.classes_)

    return {
        **cv,
        **{f'holdout_{key}': value for key, value in hold.items()},
        'estimator': estimator,
        'y_true': y_test,
        'predictions': pred_test,
        'scores': scores_test,
        'classes': estimator.classes_
    }


### Verificación del campo experimental

In [ ]:
CONFIGURATIONS=all_configurations()
config_table=pd.DataFrame([c.__dict__ for c in CONFIGURATIONS])
print('Configuraciones por dataset:', len(CONFIGURATIONS))
print('Configuraciones totales:', len(CONFIGURATIONS)*len(DATASETS))
display(config_table.head(12))
print('Modelos:', config_table.model.unique())
print('Kernels:', config_table.kernel.unique())

### Evaluación justa

Todas las configuraciones de un dataset usan exactamente los mismos índices de entrenamiento y holdout. Dentro del entrenamiento se emplea validación cruzada de 3 folds. El discretizador de la variable objetivo se ajusta dentro de cada fold para evitar utilizar información del conjunto de validación.

Las métricas son Accuracy, F1 macro, AUC one-vs-rest y MCC. La selección se hace mediante el rango promedio de las cuatro métricas de validación cruzada; no se mezclan métricas de conjuntos diferentes.

In [ ]:
from sklearn.model_selection import train_test_split

# PRIMERA EJECUCIÓN: déjalo en False para comprobar todo con pocas configuraciones.
# Cuando esta prueba termine sin errores, cambia a True y vuelve a ejecutar esta celda.
RUN_FULL_GRID = False
RANDOM_STATE = 2021

all_results = []
all_failures = []

for dataset_name, frame in DATASETS.items():
    X = frame[FEATURES].to_numpy(float)
    y = frame[TARGET].to_numpy(float)
    indices = np.arange(len(frame))
    train_idx, holdout_idx = train_test_split(
        indices, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
    )

    configs = CONFIGURATIONS if RUN_FULL_GRID else CONFIGURATIONS[:6]
    print(f'\n{dataset_name.upper()}: {len(configs)} configuraciones')

    for pos, config in enumerate(configs, start=1):
        try:
            start = time.perf_counter()
            result = evaluate(config, X, y, train_idx, holdout_idx, RANDOM_STATE)
            elapsed = time.perf_counter() - start

            row = {
                'dataset': dataset_name,
                'configuration': config.name,
                'scaler': config.scaler,
                'discretizer': config.discretizer,
                'reducer': config.reducer,
                'model': config.model,
                'kernel': config.kernel,
                'elapsed_seconds': elapsed
            }
            row.update({
                key: value for key, value in result.items()
                if key not in {'estimator', 'y_true', 'predictions', 'scores', 'classes'}
            })
            all_results.append(row)
            print(f'  {pos}/{len(configs)} OK - {config.name} - {elapsed:.2f}s')

        except Exception as error:
            all_failures.append({
                'dataset': dataset_name,
                'configuration': config.name,
                'error': repr(error)
            })
            print(f'  {pos}/{len(configs)} ERROR - {config.name}: {error}')

results_df = pd.DataFrame(all_results)
failures_df = pd.DataFrame(all_failures)

print('\nResultados:', results_df.shape)
print('Fallos:', failures_df.shape)
if not failures_df.empty:
    display(failures_df.head(20))


### Selección de configuraciones y resultados

Para no seleccionar un modelo por una sola métrica, se calcula un rango descendente por Accuracy, F1 macro, AUC y MCC dentro de cada dataset. El promedio de esos cuatro rangos produce `selection_rank`; valores menores indican una posición más alta dentro de este procedimiento experimental.

**Antes de continuar:** si esta es la primera ejecución, verifica que la prueba corta de la celda anterior terminó correctamente. Después cambia `RUN_FULL_GRID = True` en esa celda y ejecútala nuevamente para obtener las **648 configuraciones por dataset**.


In [ ]:
rank_metrics = ['cv_accuracy_mean', 'cv_f1_macro_mean', 'cv_auc_ovr_mean', 'cv_mcc_mean']

if results_df.empty:
    raise RuntimeError('No se obtuvo ningún resultado. Revisa la tabla de fallos de la celda anterior.')

ranked = results_df.copy()
for metric in rank_metrics:
    ranked['rank_' + metric] = ranked.groupby('dataset')[metric].rank(
        method='min', ascending=False, na_option='bottom'
    )

rank_cols = ['rank_' + metric for metric in rank_metrics]
ranked['selection_rank'] = ranked[rank_cols].mean(axis=1)
ranked = ranked.sort_values(
    ['dataset', 'selection_rank', 'cv_f1_macro_mean', 'cv_mcc_mean'],
    ascending=[True, True, False, False]
)

top3 = ranked.groupby('dataset').head(3).reset_index(drop=True)
print('Tres primeras configuraciones del procedimiento de selección:')
display(top3[[
    'dataset', 'configuration', 'selection_rank',
    'cv_accuracy_mean', 'cv_f1_macro_mean', 'cv_auc_ovr_mean', 'cv_mcc_mean',
    'holdout_accuracy', 'holdout_f1_macro', 'holdout_auc_ovr', 'holdout_mcc'
]])

RESULTS_DIR = Path('/content/results')
MODELS_DIR = Path('/content/models')
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

for dataset_name in ranked.dataset.unique():
    ranked[ranked.dataset == dataset_name].to_csv(
        RESULTS_DIR / f'{dataset_name}_results.csv', index=False
    )

print('Resultados guardados en:', RESULTS_DIR)


### Matriz de confusión y curva ROC/precisión-recall

Se visualiza la configuración ubicada en la primera posición del procedimiento de selección para cada dataset. Si el conjunto está desbalanceado, se muestra precisión-recall; de lo contrario se muestra ROC.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve, average_precision_score

def plot_diagnostics(dataset_name, config):
    frame=DATASETS[dataset_name]
    X=frame[FEATURES].to_numpy(float); y=frame[TARGET].to_numpy(float)
    idx=np.arange(len(frame)); tr,ho=train_test_split(idx,test_size=.20,random_state=RANDOM_STATE,shuffle=True)
    result=evaluate(config,X,y,tr,ho,RANDOM_STATE)
    yt=result['y_true']; yp=result['predictions']; scores=result['scores']; classes=result['classes']
    fig,ax=plt.subplots(1,2,figsize=(13,4.5))
    ConfusionMatrixDisplay.from_predictions(yt,yp,labels=classes,colorbar=False,ax=ax[0])
    ax[0].set_title(f'{dataset_name.upper()} - matriz de confusión')
    counts=np.bincount(yt.astype(int)) if np.issubdtype(yt.dtype,np.integer) else np.array([])
    imbalance=(len(counts)>0 and counts.min()>0 and counts.max()/counts.min()>2)
    if imbalance:
        for j,c in enumerate(classes):
            b=(yt==c).astype(int)
            if len(np.unique(b))==2:
                p,r,_=precision_recall_curve(b,scores[:,j]); ap=average_precision_score(b,scores[:,j]); ax[1].plot(r,p,label=f'Clase {c} AP={ap:.3f}')
        ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precisión'); ax[1].set_title('Precisión-Recall')
    else:
        for j,c in enumerate(classes):
            b=(yt==c).astype(int)
            if len(np.unique(b))==2:
                fpr,tpr,_=roc_curve(b,scores[:,j]); ax[1].plot(fpr,tpr,label=f'Clase {c} AUC={auc(fpr,tpr):.3f}')
        ax[1].plot([0,1],[0,1],'--'); ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR'); ax[1].set_title('ROC one-vs-rest')
    ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

for dataset_name in ranked.dataset.unique():
    config_row=ranked[ranked.dataset==dataset_name].iloc[0]
    cfg=Configuration(config_row.scaler,config_row.discretizer,config_row.reducer,config_row.model,config_row.kernel)
    plot_diagnostics(dataset_name,cfg)

## Exportación de los modelos para el punto 5

Los ganadores se vuelven a entrenar con el 80% de entrenamiento fijo y se guardan mediante `cloudpickle` incluido en Joblib. También se conserva el nombre de la configuración y la lista de variables para que la aplicación pueda reconstruir las predicciones.

In [ ]:
# Exportar los tres primeros modelos de cada dataset para que la aplicación pueda comparar dos.
for dataset_name in ranked.dataset.unique():
    top_rows=ranked[ranked.dataset==dataset_name].head(3)
    frame=DATASETS[dataset_name]
    X=frame[FEATURES].to_numpy(float); y=frame[TARGET].to_numpy(float)
    idx=np.arange(len(frame)); tr,ho=train_test_split(idx,test_size=.20,random_state=RANDOM_STATE,shuffle=True)
    for position, (_, row) in enumerate(top_rows.iterrows(), start=1):
        cfg=Configuration(row.scaler,row.discretizer,row.reducer,row.model,row.kernel)
        fitted=IrradiancePipeline(cfg).fit(X[tr],y[tr])
        artifact={'dataset':dataset_name,'configuration':cfg.name,'features':FEATURES,'model':fitted,'position':position}
        output=MODELS_DIR/f'{dataset_name}_model_{position}.joblib'
        with output.open('wb') as f:
            joblib.externals.cloudpickle.dump(artifact,f,protocol=5)
        print('Modelo exportado:',output)


## Punto 5. Aplicación web

La aplicación se encuentra en `app/app.py` y usa Streamlit + Folium. Permite:

1. cargar los modelos almacenados;
2. visualizar puntos clasificados sobre un mapa;
3. comparar dos configuraciones mediante sus métricas;
4. consultar la clasificación asociada a un punto seleccionado del mapa.

Los modelos y resultados deben generarse primero desde este Notebook.


### Pasos para llevar los archivos a GitHub

1. Descargar las carpetas `models`, `results` y los CSV generados.
2. Copiar el Notebook y `app/app.py` al repositorio.
3. Incluir `requirements.txt` y `README.md`.
4. Probar localmente con `streamlit run app/app.py`.
5. Subir el repositorio a GitHub y usar el README como guía de despliegue.

**Importante:** no se deben publicar credenciales ni datos privados.